# Hyperparameter search with fold pruning + the scikit-learn contract

`sklm` estimators honour the scikit-learn parameter protocol, and the config objects
(`TrainingConfig`, `LoRAConfig`, `GenerationConfig`) subclass `BaseEstimator`. So **every nested
field is addressable** through the usual `__` path, and any sklearn-compatible search can tune it.
Inside a `Pipeline` step named `lm`:

- `lm__precision` — a flat field on the estimator.
- `lm__training__epochs` — the `epochs` field of the nested `TrainingConfig`.
- `lm__lora__rank` — the `rank` field of the nested `LoRAConfig`.

You declare the fixed hyperparameters once on the estimator and put only the swept fields in
`param_distributions`; there is no need to re-instantiate a whole config per trial.

## How `PruningSearchCV` searches

`sklm.tuning.PruningSearchCV` is an Optuna-backed search built for the cost profile of fine-tuning.
For each trial $t$ it proposes a configuration $\theta_t$ and evaluates it by $k$-fold
cross-validation, optimizing the mean validation accuracy

$$ s(\theta_t) = \frac{1}{k}\sum_{i=1}^{k} \operatorname{accuracy}\!\left(\text{fold } i;\, \theta_t\right). $$

The sampler is **TPE** (Tree-structured Parzen Estimator): instead of grid or random search it
models the past trials with two densities over the hyperparameters — $l(\theta)$ fit to the
better-scoring trials and $g(\theta)$ to the rest — and proposes the $\theta$ that maximizes the
ratio $l(\theta)\,/\,g(\theta)$, concentrating samples where good scores have already appeared.

Two things set it apart from `optuna.integration.OptunaSearchCV`, both motivated by how expensive
each fold is here:

- **Fold-level pruning.** After every fold it reports the running mean to the trial and consults the
  study's pruner (`MedianPruner`); a config clearly below the median stops *without fine-tuning the
  remaining folds*. (`OptunaSearchCV` can only prune along a `partial_fit` learning curve, which the
  LM estimators do not expose.)
- **A variance penalty.** Selection ranks completed trials by $\mathrm{adj} = \mathrm{mean} -
  \lambda\,\mathrm{std}$ over the folds, preferring a config that is *consistent* across folds, with
  wall-clock as a defensive tie-break.

With $k$-fold CV inside each of $T$ trials, the search fine-tunes the model up to $T \times k$ times
— fewer once pruning starts skipping folds — and that is the dominant cost. distilgpt2 on Iris is a
weak classifier; the point here is the **tuning ergonomics**, not the accuracy.

> This notebook runs `mlx-community/distilgpt2` with `n_trials=20` and `cv=4` (up to `80` fine-tunes,
> fewer with pruning). Lower `n_trials` / `cv` for a quicker pass, or raise the model for accuracy.

In [ ]:
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from optuna.distributions import CategoricalDistribution, FloatDistribution, IntDistribution
from optuna.logging import WARNING, set_verbosity
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklm import (
    Callback,
    KeyValueSerializer,
    LanguageModelClassifier,
    SpacedDigits,
    TrainingConfig,
)
from sklm.tuning import LogCallback, PruningSearchCV, best_trial

set_verbosity(WARNING)  # silence Optuna's per-trial INFO/pruned logs; LogCallback is the view

SEED = 42

## Estimator, pipeline, and search space

`SpacedDigits` serializes each number one token per digit (`5 . 1`), which can help a small model
read magnitudes. The fixed knobs live on the estimator; the swept fields — the schedule's learning
rate and warmup (one level deeper, under `lr_scheduler`), epochs, batch size, and two LoRA fields —
are addressed by their `__` paths.

In [2]:
clf = LanguageModelClassifier(
    model="mlx-community/distilgpt2",
    backend="mlx",
    precision="fp32",
    serializer=KeyValueSerializer(number=SpacedDigits()),
    training=TrainingConfig(augmentation_factor=12),
    callback=Callback(),  # silent no-op; None would auto-attach a per-fit dashboard (60 here)
    random_state=SEED,
)

pipe = Pipeline([("lm", clf)])

param_distributions = {
    "lm__training__lr_scheduler__learning_rate": CategoricalDistribution([0.00001, 0.00002, 0.00003]),
    "lm__training__lr_scheduler__warmup_ratio": FloatDistribution(0.0, 0.1),
    "lm__training__epochs": IntDistribution(3, 5),
    "lm__training__batch_size": CategoricalDistribution([16]),
}

## Watching the search

`PruningSearchCV` accepts **Optuna study callbacks** — any `Callable[[Study, FrozenTrial], None]`
passed as `callbacks=[...]`, invoked after each finished trial. `sklm.tuning.LogCallback` is one:
it appends **one plain line per CV fold and per finished trial** — no screen control, no widgets —
so the same output reads fine in a notebook, a terminal, or a CI log.

Each trial records its per-fold scores, so a completed trial's line shows `mean ±std` over the
folds next to the **risk-adjusted** score

$$\mathrm{adj} = \mathrm{mean} - \lambda\,\mathrm{std}, \qquad \lambda = \texttt{STD\_PENALTY}.$$

Penalizing the fold-to-fold std prefers a config that is *consistent* across folds over one that is
equally good on average but erratic. Every completed line says where the trial landed — `★ new
best` or a pointer to the current best (exactly what `best_trial` refits below) — while a **pruned**
trial's line shows the running mean and the fold it stopped at.

The penalty is a **selection** re-ranking, not a change to the search: TPE still proposes trials by
mean score, so a low-variance region it never samples won't be found — `adj` only re-ranks what was
tried. Raise λ to lean harder on stability, set it to `0` to recover the plain-mean ranking. With
only 4 folds `std` is a noisy estimate, so keep λ modest.

## Run the search

`PruningSearchCV` clones the pipeline for every (trial, fold) fit, so each fine-tune starts from the
base model with no state leaking between trials. The sampler and pruner are passed explicitly here;
`std_penalty` drives both the leaderboard's `adj` column and the final selection. With `refit=True`
(the default) the winning config is refit on the full training split into `best_estimator_`.

In [ ]:
STD_PENALTY = 0.5  # lambda in adj = mean - lambda*std: how hard fold-to-fold variance is penalized

search = PruningSearchCV(
    pipe,
    param_distributions,
    cv=10,
    scoring="accuracy",
    n_trials=10,
    std_penalty=STD_PENALTY,
    sampler=TPESampler(multivariate=True, seed=SEED),
    pruner=MedianPruner(),
    callbacks=[LogCallback(std_penalty=STD_PENALTY)],
    random_state=SEED,
)

iris = load_iris(as_frame=True)
X = iris.data
y = iris.target_names[iris.target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

search.fit(X_train, y_train);

## Best configuration

`PruningSearchCV` already refit the selected config into `best_estimator_` — the trial with the
highest risk-adjusted `mean - λ·std`, ties broken toward the faster trial. Ranking the same study
with `std_penalty=0` selects by mean alone, so the two can disagree once variance is penalized.

In [4]:
best = search.best_trial_
attrs = best.user_attrs
mean_only = best_trial(search.study_, std_penalty=0.0)
print(
    f"selected trial:   #{best.number}  "
    f"(mean {attrs['mean_test_score']:.3f} +/- {attrs['std_test_score']:.3f}, "
    f"adj {search.best_score_:.3f}, {best.duration.total_seconds():.0f}s)"
)
print(f"mean-only best:   #{mean_only.number}  (std_penalty=0, by mean alone)")
print(f"best params:      {best.params}")
print(f"test accuracy:    {accuracy_score(y_test, search.predict(X_test)):.3f}")

selected trial:   #3  (mean 0.967 +/- 0.041, adj 0.946, 153s)
mean-only best:   #3  (std_penalty=0, by mean alone)
best params:      {'lm__training__lr_scheduler__learning_rate': 3e-05, 'lm__training__lr_scheduler__warmup_ratio': 0.04319450186421158, 'lm__training__epochs': 3, 'lm__training__batch_size': 16}
test accuracy:    0.900
